# mHC / HC / Baseline Training + Benchmark on Google Colab T4

Notebook này thực hiện workflow đầy đủ:

1. Kiểm tra GPU T4/CUDA.
2. Mount Google Drive để lưu checkpoint và report.
3. Clone repo mHC implementation.
4. Cài dependencies.
5. Download FineWeb10B GPT-2 shards.
6. Smoke test pipeline.
7. Train 3 biến thể từ scratch:
   - baseline Transformer residual
   - traditional HC
   - mHC
8. Benchmark inference cho 3 checkpoint.
9. Summarize training + benchmark.
10. Vẽ chart phục vụ visualization/result comparison/summary.

> Ghi chú quan trọng: Đây là small-scale controlled comparison trên nanoGPT + FineWeb10B + T4. Không claim reproduce toàn bộ paper-scale benchmark như MMLU/BBH/GSM8K nếu bạn chưa tự implement và chạy các benchmark đó.

## 0. Runtime requirement

Trước khi chạy notebook:

`Runtime → Change runtime type → Hardware accelerator: GPU`

Khuyến nghị:
- GPU: NVIDIA T4 16GB
- dtype: `float16`
- Colab Free: nên bắt đầu với `MAX_ITERS=2000` hoặc `3000`
- Colab Pro/Pro+: có thể dùng `MAX_ITERS=5000`

In [1]:
# Check GPU
!nvidia-smi

import torch, os, platform, sys
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())

Wed May 20 06:29:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive

Drive dùng để lưu checkpoint/report sau khi train. Train trực tiếp trên `/content` sẽ nhanh hơn, sau đó notebook sẽ copy kết quả sang Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Global configuration

Bạn cần sửa `REPO_URL` thành repo GitHub của bạn.

Ví dụ:

```python
REPO_URL = "https://github.com/khoanguyen/mHC-manifold-constrained-hyper-connections.git"
```

Nếu bạn chưa push repo lên GitHub, hãy upload zip repo vào Colab và chỉnh phần clone ở cell sau.

In [4]:
from pathlib import Path
import os, json, shutil, subprocess, time, textwrap, glob

# =========================
# EDIT THESE VALUES
# =========================

REPO_URL = "https://github.com/khoaoe/mHC-manifold-constrained-hyper-connections.git"  
BRANCH = "t4-inference-benchmark"  

# Dataset shards:
# - 1: smoke/very quick
# - 9: recommended for Colab T4
# - 20: better, but uses more RAM/disk
DATA_SHARDS = 9

# Training scale:
# - Smoke: 2-20
# - Colab Free realistic: 2000-3000
# - Full intended T4 comparison: 5000
MAX_ITERS = 3000
EVAL_INTERVAL = 300
EVAL_ITERS = 50

# T4-safe defaults
BATCH_SIZE = 8
GRAD_ACCUM = 8
DTYPE = "float16"
DEVICE = "cuda"
WANDB_LOG = "False"

# Benchmark settings
BENCH_BATCH_SIZE = 1
PROMPT_LEN = 128
GEN_LEN = 32
NUM_WARMUP = 5
NUM_ITERS = 20
COMPILE = "false"

# Run naming / paths
RUN_NAME = f"t4-fineweb{DATA_SHARDS}shards-{MAX_ITERS}iters"
DRIVE_ROOT = Path("/content/drive/MyDrive/mhc")
DRIVE_RUNS = DRIVE_ROOT / "runs" / RUN_NAME
DRIVE_REPORTS = DRIVE_ROOT / "reports" / RUN_NAME

LOCAL_REPO = Path("/content/mhc")
NANOGPT_DIR = LOCAL_REPO / "examples" / "nanogpt"

print("RUN_NAME:", RUN_NAME)
print("DRIVE_RUNS:", DRIVE_RUNS)
print("DRIVE_REPORTS:", DRIVE_REPORTS)

RUN_NAME: t4-fineweb9shards-3000iters
DRIVE_RUNS: /content/drive/MyDrive/mhc/runs/t4-fineweb9shards-3000iters
DRIVE_REPORTS: /content/drive/MyDrive/mhc/reports/t4-fineweb9shards-3000iters


## 3. Clone repo and install dependencies

Nếu cell này báo lỗi vì `REPO_URL` vẫn còn placeholder, hãy sửa ở cell config phía trên.

In [5]:
%cd /content

if str(REPO_URL).startswith("https://github.com/<"):
    raise ValueError("Bạn cần sửa REPO_URL trong cell cấu hình trước khi clone repo.")

if LOCAL_REPO.exists():
    print(f"Repo already exists at {LOCAL_REPO}. Pulling latest changes...")
    %cd /content/mhc
    !git fetch --all
    !git checkout {BRANCH}
    !git pull
else:
    !git clone --branch {BRANCH} {REPO_URL} {LOCAL_REPO}

%cd /content/mhc
!git status --short
!git rev-parse HEAD

/content
Cloning into '/content/mhc'...
remote: Enumerating objects: 470, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 470 (delta 181), reused 270 (delta 143), pack-reused 142 (from 1)
Receiving objects: 100% (470/470), 215.69 KiB | 7.99 MiB/s, done.
Resolving deltas: 100% (262/262), done.
/content/mhc
67c06afc30f8327cc510c5b0d153f6b3b97dab51


In [6]:
%cd /content/mhc

# Install project dependencies.
# The repo pyproject includes torch/einops/numpy/tiktoken/wandb/pytest;
# [examples] adds huggingface-hub for FineWeb shard download.
!pip install -q -e ".[examples]"
!pip install -q pandas matplotlib

# Optional sanity tests for helper scripts. These should be quick.
!python -m pytest -q tests/test_benchmark_summary.py tests/test_training_summary.py tests/test_nanogpt_inference_cli.py

/content/mhc
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 13.1 MB/s eta 0:00:0000:01m00:01
  Building editable for mhc-hyper-connections (pyproject.toml) ... done
.......                                                                  [100%]
7 passed in 11.07s


## 4. Download FineWeb10B GPT-2 shards

Mặc định script tải:
- 1 validation shard
- `DATA_SHARDS` training shards

Với T4/Colab, `DATA_SHARDS=9` là điểm cân bằng tốt. Đừng tải full 103 shards trên Colab Free nếu không thật sự cần.

In [7]:
%cd /content/mhc

!python examples/nanogpt/data/fineweb10B/download.py {DATA_SHARDS}

# Inspect dataset files
!ls -lh examples/nanogpt/data/fineweb10B | head -20
!du -sh examples/nanogpt/data/fineweb10B

/content/mhc
  - 1 validation shard
  - 9 training shards

  downloading fineweb_val_000000.bin...
fineweb_val_000000.bin: 100% 200M/200M [00:02<00:00, 70.8MB/s]
  downloading fineweb_train_000001.bin...
fineweb_train_000001.bin: 100% 200M/200M [00:01<00:00, 110MB/s] 
  downloading fineweb_train_000002.bin...
fineweb_train_000002.bin: 100% 200M/200M [00:06<00:00, 33.3MB/s] 
  downloading fineweb_train_000003.bin...
fineweb_train_000003.bin: 100% 200M/200M [00:03<00:00, 52.3MB/s]
  downloading fineweb_train_000004.bin...
fineweb_train_000004.bin: 100% 200M/200M [00:04<00:00, 41.6MB/s] 
  downloading fineweb_train_000005.bin...
fineweb_train_000005.bin: 100% 200M/200M [00:05<00:00, 39.9MB/s]
  downloading fineweb_train_000006.bin...
fineweb_train_000006.bin: 100% 200M/200M [00:02<00:00, 90.4MB/s]
  downloading fineweb_train_000007.bin...
fineweb_train_000007.bin: 100% 200M/200M [00:02<00:00, 99.5MB/s]
  downloading fineweb_train_000008.bin...
fineweb_train_000008.bin: 100% 200M/200M [00:

## 5. Smoke test

Chạy 2 iterations baseline để kiểm tra:
- CUDA
- data loader
- model forward/backward
- checkpoint/log files

Output smoke test không dùng để claim model quality.

In [8]:
%cd /content/mhc/examples/nanogpt

SMOKE_OUT = "out-smoke-baseline"

!python train.py config/train_fineweb10B_t4.py \
  "out_dir='{SMOKE_OUT}'" \
  "max_iters=2" \
  "eval_interval=1" \
  "eval_iters=1" \
  "batch_size=2" \
  "gradient_accumulation_steps=1" \
  "block_size=128" \
  "device='cuda'" \
  "dtype='float16'" \
  "wandb_log=False" \
  "compile_model=False"

!ls -lh {SMOKE_OUT}
!cat {SMOKE_OUT}/summary.json

/content/mhc/examples/nanogpt
Found 9 train shards, 1 val shards
^C
total 16K
-rwxr-xr-x 1 root root  351 May 20 06:34 command.sh
-rw-r--r-- 1 root root 1.5K May 20 06:34 config_effective.json
-rw-r--r-- 1 root root 1.8K May 20 06:34 dataset_manifest.json
-rw-r--r-- 1 root root 1.1K May 20 06:34 run_metadata.json
cat: out-smoke-baseline/summary.json: No such file or directory


## 6. Helpers: train one variant, sync to Drive, verify outputs

Các hàm dưới đây giúp train từng variant một. Cách này an toàn hơn `run_t4_full_compare.sh` trên Colab Free vì sau mỗi variant notebook sẽ copy checkpoint/log sang Drive.

In [9]:
import subprocess, shutil, os, json, time
from pathlib import Path

VARIANTS = {
    "baseline": {
        "config": "config/train_fineweb10B_t4.py",
        "out_dir": "out-t4-baseline",
    },
    "hc": {
        "config": "config/train_fineweb10B_hc_t4.py",
        "out_dir": "out-t4-hc",
    },
    "mhc": {
        "config": "config/train_fineweb10B_mhc_t4.py",
        "out_dir": "out-t4-mhc",
    },
}

def run_cmd(cmd, cwd=None, env=None):
    print("\n$ " + " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(map(str, cmd))}")
    return "".join(lines)

def train_variant(name, *, force=False):
    assert name in VARIANTS, f"Unknown variant: {name}"
    cfg = VARIANTS[name]
    out_dir = NANOGPT_DIR / cfg["out_dir"]
    drive_out = DRIVE_RUNS / cfg["out_dir"]

    if out_dir.exists() and (out_dir / "ckpt.pt").exists() and not force:
        print(f"[SKIP] Local checkpoint exists: {out_dir / 'ckpt.pt'}")
        return out_dir

    if drive_out.exists() and (drive_out / "ckpt.pt").exists() and not out_dir.exists():
        print(f"[RESTORE] Copying existing run from Drive: {drive_out} -> {out_dir}")
        shutil.copytree(drive_out, out_dir)

    if out_dir.exists() and (out_dir / "ckpt.pt").exists() and not force:
        print(f"[SKIP] Restored checkpoint exists: {out_dir / 'ckpt.pt'}")
        return out_dir

    cmd = [
        "python", "train.py", cfg["config"],
        f"out_dir='{cfg['out_dir']}'",
        f"max_iters={MAX_ITERS}",
        f"eval_interval={EVAL_INTERVAL}",
        f"eval_iters={EVAL_ITERS}",
        f"batch_size={BATCH_SIZE}",
        f"gradient_accumulation_steps={GRAD_ACCUM}",
        f"device='{DEVICE}'",
        f"dtype='{DTYPE}'",
        f"wandb_log={WANDB_LOG}",
        "compile_model=False",
    ]

    start = time.time()
    run_cmd(cmd, cwd=NANOGPT_DIR)
    print(f"[DONE] {name} elapsed: {(time.time() - start)/60:.2f} minutes")

    sync_variant_to_drive(name)
    return out_dir

def sync_variant_to_drive(name):
    cfg = VARIANTS[name]
    out_dir = NANOGPT_DIR / cfg["out_dir"]
    drive_out = DRIVE_RUNS / cfg["out_dir"]

    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
    if drive_out.exists():
        shutil.rmtree(drive_out)
    shutil.copytree(out_dir, drive_out)
    print(f"[SYNC] {out_dir} -> {drive_out}")

def verify_variant(name):
    cfg = VARIANTS[name]
    out_dir = NANOGPT_DIR / cfg["out_dir"]
    required = [
        "ckpt.pt",
        "summary.json",
        "metrics.jsonl",
        "config_effective.json",
        "dataset_manifest.json",
        "run_metadata.json",
        "command.sh",
    ]
    print(f"\n== {name}: {out_dir} ==")
    ok = True
    for fname in required:
        path = out_dir / fname
        exists = path.exists()
        ok = ok and exists
        print(f"{'OK' if exists else 'MISSING'}  {fname}")
    if (out_dir / "summary.json").exists():
        with open(out_dir / "summary.json", "r", encoding="utf-8") as f:
            summary = json.load(f)
        print(json.dumps({
            "ok": summary.get("ok"),
            "best_val_loss": summary.get("best_val_loss"),
            "last_eval": summary.get("last_eval"),
            "tokens_seen": summary.get("tokens_seen"),
            "iter_num": summary.get("iter_num"),
            "elapsed_s": summary.get("elapsed_s"),
            "device": summary.get("device"),
            "dtype": summary.get("dtype"),
        }, indent=2))
    return ok

## 7. Train from scratch: baseline → HC → mHC

Cell này có thể chạy lâu. Nếu Colab disconnect, chạy lại notebook từ đầu; helper sẽ restore run đã sync từ Drive nếu checkpoint đã tồn tại.

In [10]:
%cd /content/mhc

# Train sequentially. Change force=True only when you intentionally want to retrain.
for variant in ["baseline", "hc", "mhc"]:
    train_variant(variant, force=False)
    verify_variant(variant)

/content/mhc

$ python train.py config/train_fineweb10B_t4.py out_dir='out-t4-baseline' max_iters=3000 eval_interval=300 eval_iters=50 batch_size=8 gradient_accumulation_steps=8 device='cuda' dtype='float16' wandb_log=False compile_model=False


RuntimeError: Command failed with exit code -9: python train.py config/train_fineweb10B_t4.py out_dir='out-t4-baseline' max_iters=3000 eval_interval=300 eval_iters=50 batch_size=8 gradient_accumulation_steps=8 device='cuda' dtype='float16' wandb_log=False compile_model=False

## 8. Summarize training runs

Tạo:
- `training_summary.csv`
- `training_summary.md`
- `training_summary.json`

Các file được lưu trong `DRIVE_REPORTS`.

In [ ]:
%cd /content/mhc

DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)

!python examples/nanogpt/summarize_training_runs.py \
  --runs \
  "baseline=examples/nanogpt/out-t4-baseline" \
  "hc=examples/nanogpt/out-t4-hc" \
  "mhc=examples/nanogpt/out-t4-mhc" \
  --output-dir "{DRIVE_REPORTS}"

!ls -lh "{DRIVE_REPORTS}"
!cat "{DRIVE_REPORTS}/training_summary.md"

## 9. Benchmark inference

Benchmark dùng synthetic token IDs để đo runtime:
- prefill latency
- full-context generation latency
- latency/token
- tokens/sec
- peak VRAM

Đây là runtime benchmark, không phải benchmark chất lượng ngôn ngữ.

In [ ]:
%cd /content/mhc

BENCH_DIR = DRIVE_REPORTS / "benchmarks"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

!CKPT_BASELINE=examples/nanogpt/out-t4-baseline/ckpt.pt \
CKPT_HC=examples/nanogpt/out-t4-hc/ckpt.pt \
CKPT_MHC=examples/nanogpt/out-t4-mhc/ckpt.pt \
OUT_DIR="{BENCH_DIR}" \
DEVICE=cuda DTYPE=float16 BATCH_SIZE={BENCH_BATCH_SIZE} PROMPT_LEN={PROMPT_LEN} GEN_LEN={GEN_LEN} \
NUM_WARMUP={NUM_WARMUP} NUM_ITERS={NUM_ITERS} COMPILE={COMPILE} \
bash examples/nanogpt/run_t4_benchmarks.sh

!python examples/nanogpt/summarize_benchmarks.py "{BENCH_DIR}"
!cat "{BENCH_DIR}/summary.md"

## 10. Text generation demo

Dùng checkpoint mHC để generate một đoạn text. Chỉ dùng phần này làm demo inference. Nếu model train ít iteration thì output có thể chưa hay.

In [ ]:
%cd /content/mhc

!python examples/nanogpt/infer.py \
  --ckpt examples/nanogpt/out-t4-mhc/ckpt.pt \
  --config examples/nanogpt/config/train_fineweb10B_mhc_t4.py \
  --device cuda \
  --dtype float16 \
  --compile false \
  --prompt "The future of machine learning is" \
  --max-new-tokens 64 \
  --temperature 0.8 \
  --top-k 200 \
  --seed 1337

## 11. Visualization: training curves

Cell này đọc `metrics.jsonl` của 3 variants và tạo chart PNG trong `DRIVE_REPORTS/figures`.

Các chart hữu ích cho slide:
- train loss curve
- validation loss curve
- tokens/sec curve
- peak VRAM curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

FIG_DIR = DRIVE_REPORTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def read_metrics(variant, run_dir):
    path = Path(run_dir) / "metrics.jsonl"
    if not path.exists():
        print(f"[WARN] missing metrics: {path}")
        return pd.DataFrame()
    df = pd.read_json(path, lines=True)
    df["variant"] = variant
    return df

frames = []
for variant, cfg in VARIANTS.items():
    frames.append(read_metrics(variant, NANOGPT_DIR / cfg["out_dir"]))
metrics = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

display(metrics.head())
print(metrics["event"].value_counts(dropna=False) if not metrics.empty else "No metrics loaded")

def plot_metric(event, metric, ylabel, filename):
    if metrics.empty:
        print("No metrics available")
        return
    df = metrics[metrics["event"] == event].copy()
    if metric not in df.columns:
        print(f"[WARN] metric {metric} not found")
        return
    df = df.dropna(subset=[metric, "iter"])
    if df.empty:
        print(f"[WARN] no rows for {event}/{metric}")
        return

    plt.figure(figsize=(9, 5))
    for variant in ["baseline", "hc", "mhc"]:
        sub = df[df["variant"] == variant].sort_values("iter")
        if len(sub):
            plt.plot(sub["iter"], sub[metric], marker="o", label=variant)
    plt.xlabel("Iteration")
    plt.ylabel(ylabel)
    plt.title(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    out_path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.show()
    print("saved:", out_path)

plot_metric("train", "train_loss", "Training loss", "train_loss_curve.png")
plot_metric("eval", "val_loss", "Validation loss", "val_loss_curve.png")
plot_metric("train", "tokens_per_sec", "Training tokens/sec", "tokens_per_sec_curve.png")
plot_metric("train", "peak_vram_mb", "Peak VRAM during training (MB)", "peak_vram_curve.png")

## 12. Visualization: summary bar charts

Cell này tạo chart từ:
- `training_summary.csv`
- `benchmarks/summary.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

training_csv = DRIVE_REPORTS / "training_summary.csv"
benchmark_csv = DRIVE_REPORTS / "benchmarks" / "summary.csv"

train_summary = pd.read_csv(training_csv) if training_csv.exists() else pd.DataFrame()
bench_summary = pd.read_csv(benchmark_csv) if benchmark_csv.exists() else pd.DataFrame()

display(train_summary)
display(bench_summary)

def bar_from_df(df, x, y, title, ylabel, filename):
    if df.empty or y not in df.columns:
        print(f"[WARN] cannot plot {filename}")
        return
    plot_df = df[[x, y]].copy()
    plot_df[y] = pd.to_numeric(plot_df[y], errors="coerce")
    plot_df = plot_df.dropna(subset=[y])
    if plot_df.empty:
        print(f"[WARN] no numeric data for {y}")
        return

    plt.figure(figsize=(8, 5))
    plt.bar(plot_df[x], plot_df[y])
    plt.xlabel(x)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(axis="y", alpha=0.3)
    out_path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.show()
    print("saved:", out_path)

bar_from_df(train_summary, "variant", "best_val_loss", "Best validation loss", "loss", "best_val_loss_bar.png")
bar_from_df(train_summary, "variant", "final_val_ppl", "Final validation perplexity", "perplexity", "final_val_ppl_bar.png")
bar_from_df(bench_summary, "variant", "tokens_per_sec_mean", "Inference throughput", "tokens/sec", "inference_tokens_per_sec_bar.png")
bar_from_df(bench_summary, "variant", "full_context_generation_latency_per_token_ms_mean", "Generation latency per token", "ms/token", "decode_ms_per_token_bar.png")
bar_from_df(bench_summary, "variant", "peak_vram_mb_max", "Peak VRAM during benchmark", "MB", "benchmark_peak_vram_bar.png")

## 13. Final artifact checklist

Cell này kiểm tra toàn bộ file cần lấy cho report/slides.

In [ ]:
from pathlib import Path

required_paths = [
    DRIVE_REPORTS / "training_summary.csv",
    DRIVE_REPORTS / "training_summary.md",
    DRIVE_REPORTS / "training_summary.json",
    DRIVE_REPORTS / "benchmarks" / "summary.csv",
    DRIVE_REPORTS / "benchmarks" / "summary.md",
    DRIVE_REPORTS / "figures" / "train_loss_curve.png",
    DRIVE_REPORTS / "figures" / "val_loss_curve.png",
    DRIVE_REPORTS / "figures" / "best_val_loss_bar.png",
    DRIVE_REPORTS / "figures" / "inference_tokens_per_sec_bar.png",
]

print("=== Report files ===")
all_ok = True
for p in required_paths:
    ok = p.exists()
    all_ok = all_ok and ok
    print(f"{'OK' if ok else 'MISSING'}  {p}")

print("\n=== Run directories ===")
for variant, cfg in VARIANTS.items():
    local = NANOGPT_DIR / cfg["out_dir"]
    drive = DRIVE_RUNS / cfg["out_dir"]
    print(f"\n[{variant}]")
    print("local:", local, "ckpt:", (local / "ckpt.pt").exists())
    print("drive:", drive, "ckpt:", (drive / "ckpt.pt").exists())

print("\nREADY FOR REPORT:", all_ok)
print("DRIVE_REPORTS:", DRIVE_REPORTS)
print("DRIVE_RUNS:", DRIVE_RUNS)

## 14. Emergency commands

Nếu bị CUDA OOM:

1. Giảm batch, tăng grad accumulation để giữ effective batch:
```python
BATCH_SIZE = 4
GRAD_ACCUM = 16
```

2. Nếu vẫn OOM, giảm block size trong lệnh train:
```bash
"block_size=512"
```
Nhưng phải áp dụng giống nhau cho baseline/HC/mHC.

3. Nếu Colab sắp hết runtime, giảm:
```python
MAX_ITERS = 2000
EVAL_INTERVAL = 250
EVAL_ITERS = 30
```

Khi trình bày, nói đúng số iterations/dataset shards bạn đã chạy.